In [1]:
# ╔══════════════════════════════════════════════════════════╗
# ║     SimuHome — Mistral 7B Fine-Tuning Notebook          ║
# ║     Platform : Kaggle (T4 x2 GPU)                       ║
# ║     Time     : ~45-60 minutes                           ║
# ╚══════════════════════════════════════════════════════════╝

# ── CELL 1: Check GPU ────────────────────────────────────────
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)

import torch
print(f"CUDA available : {torch.cuda.is_available()}")
print(f"GPU count      : {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}         : {torch.cuda.get_device_name(i)}")
    mem = torch.cuda.get_device_properties(i).total_memory
    print(f"  VRAM        : {mem/1e9:.1f} GB")

Thu Mar 19 07:06:36 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [21]:
# ── CELL 2: Install Compatible Packages ──────────────────────
import torch
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.version.cuda}")

# Remove everything first
!pip uninstall bitsandbytes transformers peft trl accelerate -y -q

# Install bitsandbytes built for CUDA 12.8
!pip install bitsandbytes \
    --index-url https://download.pytorch.org/whl/cu128 -q

# If above fails, install from source
!pip install \
    "bitsandbytes @ git+https://github.com/TimDettmers/bitsandbytes.git@main" \
    -q 2>/dev/null || \
pip install bitsandbytes --upgrade -q

# Install compatible versions of other packages
!pip install transformers==4.44.0 -q
!pip install peft==0.12.0 -q
!pip install trl==0.9.6 -q
!pip install accelerate==0.33.0 -q
!pip install datasets==2.20.0 -q
!pip install jsonlines -q

print("✅ Done installing")

PyTorch : 2.10.0+cu128
CUDA    : 12.8


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


ERROR: Could not find a version that satisfies the requirement bitsandbytes (from versions: none)
ERROR: No matching distribution found for bitsandbytes


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


...

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 315.1/315.1 kB 6.6 MB/s eta 0:00:00a 0:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchaudio 2.9.0+cu126 requires torch==2.9.0, but you have torch 2.10.0 which is incompatible.
torchvision 0.24.0+cu126 requires torch==2.9.0, but you have torch 2.10.0 which is incompatible.
fastai 2.8.6 requires torch<2.10,>=1.10, but you have torch 2.10.0 which is incompatible.


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 547.8/547.8 kB 9.9 MB/s eta 0:00:00ta 0:00:01


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


✅ Done installing


In [3]:
# ── CELL 3: Load Training Data ───────────────────────────────
import jsonlines
import os

# Path to your uploaded dataset on Kaggle
# After uploading, Kaggle mounts it at /kaggle/input/<dataset-name>/
DATA_DIR = "//kaggle/input/datasets/jeevandasjd/simuhome-processed-data/data/processed"

# If zip was uploaded, extract first:
# !unzip /kaggle/input/simuhome-finetune/simuhome_training_data.zip -d /kaggle/working/

# Load texts
def load_texts(filepath):
    texts = []
    with jsonlines.open(filepath) as r:
        for item in r:
            texts.append(item["text"])
    return texts

train_texts = load_texts(f"{DATA_DIR}/train.jsonl")
val_texts   = load_texts(f"{DATA_DIR}/val.jsonl")
test_texts  = load_texts(f"{DATA_DIR}/test.jsonl")

print(f"Train : {len(train_texts)}")
print(f"Val   : {len(val_texts)}")
print(f"Test  : {len(test_texts)}")

# Check sample
print("\n=== Sample (first 300 chars) ===")
print(train_texts[0][:300])

Train : 481
Val   : 60
Test  : 61

=== Sample (first 300 chars) ===
<s>[INST] <<SYS>>
You are a Smart Home Assistant that uses tools to control devices
and provide information based on the Matter protocol, with the goal of fulfilling the User Query.

You operate under the ReAct framework with structured responses.

[REACT FRAMEWORK]
- LOOP: ('thought' -> 'action' ->


In [11]:
# ── CELL 4: Check Token Lengths ──────────────────────────────
# Important: sequences too long will cause OOM errors

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "mistralai/Mistral-7B-Instruct-v0.3",
    trust_remote_code=True
)
tokenizer.pad_token = tokenizer.eos_token

lengths = [len(tokenizer.encode(t)) for t in train_texts]

print(f"Min tokens : {min(lengths)}")
print(f"Max tokens : {max(lengths)}")
print(f"Avg tokens : {sum(lengths)/len(lengths):.0f}")
print(f"Over 2048  : {sum(1 for l in lengths if l > 2048)}")
print(f"Over 4096  : {sum(1 for l in lengths if l > 4096)}")

# Distribution
buckets = [512, 1024, 2048, 4096]
for b in buckets:
    count = sum(1 for l in lengths if l <= b)
    print(f"  Under {b:4d} tokens: {count}/{len(lengths)}")

Min tokens : 671
Max tokens : 2585
Avg tokens : 1162
Over 2048  : 14
Over 4096  : 0
  Under  512 tokens: 0/481
  Under 1024 tokens: 190/481
  Under 2048 tokens: 467/481
  Under 4096 tokens: 481/481


In [20]:
# ── CELL 5 FALLBACK: Load in 8-bit instead ───────────────────
# Use this if 4-bit keeps failing

import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training
)

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    "mistralai/Mistral-7B-Instruct-v0.3"
)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Loading Mistral 7B in 8-bit (fallback)...")
model = AutoModelForCausalLM.from_pretrained(
    "mistralai/Mistral-7B-Instruct-v0.3",
    load_in_8bit=True,          # 8-bit instead of 4-bit
    device_map="auto",
    torch_dtype=torch.float16,
)
model.config.use_cache = False

print("✅ Model loaded in 8-bit")
print(f"GPU memory: {torch.cuda.memory_allocated()/1e9:.2f} GB")

Loading tokenizer...


The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


Loading Mistral 7B in 8-bit (fallback)...


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

AttributeError: 'NoneType' object has no attribute 'cint8_vector_quant'